In [1]:
import requests
import time
import uuid

INSTANCE_A = "http://136.116.11.209:8080"
INSTANCE_B = "http://34.140.129.0:8080"

In [2]:
# Latency Measurement
def measure_latency(base_url, endpoint, method="GET", body_func=None, n=26):
    latencies = []
    for i in range(n):
        start = time.time()
        if method == "POST":
            # Generate the dynamic body for the current iteration (if a function is provided)
            body = body_func(i) if body_func else None
            requests.post(f"{base_url}{endpoint}", json=body)
        else:
            requests.get(f"{base_url}{endpoint}")
        latencies.append((time.time() - start) * 1000)  # ms
    return sum(latencies) / len(latencies)


def part_a():
    print("PART A: Latency Measurement")

    # Helper function to dynamically create usernames: User_A, User_B, ..., User_Z
    def generate_user(i):
        # ord('A') is 65. i % 26 ensures it safely loops back to 'A' if n > 26
        letter = chr(ord('A') + (i % 26)) 
        return {"username": f"User_{letter}"}

    # Pass the generate_user function instead of a hardcoded dictionary
    a_register = measure_latency(INSTANCE_A, "/register", "POST", body_func=generate_user, n=26)
    b_register = measure_latency(INSTANCE_B, "/register", "POST", body_func=generate_user, n=26)
    
    a_list = measure_latency(INSTANCE_A, "/list", n=26)
    b_list = measure_latency(INSTANCE_B, "/list", n=26)

    print(f"Instance A (us-central1)  /register avg: {a_register:.5f} ms")
    print(f"Instance B (europe-west1) /register avg: {b_register:.5f} ms")
    print(f"Instance A (us-central1)  /list avg:     {a_list:.5f} ms")
    print(f"Instance B (europe-west1) /list avg:     {b_list:.5f} ms")

    return a_register, b_register, a_list, b_list

In [3]:
# Observing Eventual Consistency
def part_b():
    print("\nPART B: Eventual Consistency Test")
    miss_count = 0

    for i in range(200):
        username = f"user_{uuid.uuid4().hex[:8]}_{i}"

        # Register on Instance A
        requests.post(f"{INSTANCE_A}/register", json={"username": username})

        # Immediately check Instance B
        response = requests.get(f"{INSTANCE_B}/list")
        users = response.json().get("users", [])

        if username not in users:
            miss_count += 1

    print(f"Username not found immediately: {miss_count}/200 times")
    return miss_count

In [ ]:
# Clear both instances first for clean results
requests.post(f"{INSTANCE_A}/clear")
requests.post(f"{INSTANCE_B}/clear")

a_register, b_register, a_list, b_list = part_a()
miss_count = part_b()

# Save results to Analysis.txt
with open("Analysis_PartIV.txt", "w") as f:
    f.write("=== Part IV-A: Latency Results ===\n")
    f.write(f"Instance A (us-central1)  /register avg: {a_register:.5f} ms\n")
    f.write(f"Instance B (europe-west1) /register avg: {b_register:.5f} ms\n")
    f.write(f"Instance A (us-central1)  /list avg:     {a_list:.5f} ms\n")
    f.write(f"Instance B (europe-west1) /list avg:     {b_list:.5f} ms\n")
    f.write("\n=== Part IV-B: Eventual Consistency ===\n")
    f.write(f"Username not found immediately: {miss_count}/200 times\n")
    f.write("\n=== Discussion ===\n")

print("\nAnalysis.txt saved!")

PART A: Latency Measurement
Instance A (us-central1)  /register avg: 121.62538 ms
Instance B (europe-west1) /register avg: 458.11193 ms
Instance A (us-central1)  /list avg:     106.26866 ms
Instance B (europe-west1) /list avg:     396.31871 ms

PART B: Eventual Consistency Test
Username not found immediately: 0/200 times

Analysis.txt saved!


> curl http://136.116.11.209:8080/list   
{"users":["User_A","User_B","User_C","User_D","User_E","User_F","User_G","User_H","User_I","User_J","User_K","User_L","User_M","User_N","User_O","User_P","User_Q","User_R","User_S","User_T","User_U","User_V","User_W","User_X","User_Y","User_Z","user_012525cf_21","user_0324142a_190","user_0363f991_5","user_04d1b674_75","user_06a9bd68_169","user_0709e7c1_178","user_07b0fd02_122","user_08b88e89_143","user_0a7df8ee_132","user_0bb250ae_33","user_0f3902b7_25","user_0fdfc083_160","user_1112b7c8_49","user_11276b46_118","user_11f351f0_168","user_12051292_176","user_120a813f_154","user_13e37a9b_78","user_16edb478_11","user_174d80c6_113","user_1873cec1_134","user_1ccd0d22_50","user_1ededd41_157","user_1fbd04cf_61","user_208bf2d9_192","user_238aa222_174","user_27b7c738_197","user_29678262_38","user_2ae55361_140","user_2ccf47e5_112","user_2ce140de_40","user_2e407c99_90","user_2e9b01d3_191","user_2fb1b38b_139","user_33dbd7d0_183","user_34f2050f_41","user_36946b19_127","user_36ce0b3e_77","user_37b38f0f_177","user_38981ff0_164","user_3944a40e_110","user_3a122772_126","user_3bc9c0e2_158","user_3c962289_13","user_3d626baf_128","user_41cb33cb_82","user_465f6abf_71","user_46ac8199_88","user_491dd3dd_171","user_4a091785_6","user_4af40dbb_68","user_4b26384f_188","user_4b85f2e7_125","user_4bbdedb8_156","user_4cd0ebd4_116","user_4d5988eb_93","user_4da2eb1f_16","user_4e626e1e_73","user_505b82cc_152","user_50cc7b7d_43","user_50f784d4_100","user_5125cdf6_102","user_51ac7591_149","user_536bf271_67","user_540162f1_114","user_5402b77d_165","user_549c36fb_79","user_55513b50_86","user_57550d26_80","user_584e66f4_62","user_58a9fbca_98","user_5a83367e_124","user_5ace1046_57","user_5b51bc0d_47","user_5bc55185_14","user_5bcbbe66_18","user_5d2522ab_12","user_5dcc1a1d_81","user_61159ea7_64","user_611fa5be_182","user_63dbcefd_111","user_63f67e13_115","user_645a05f7_99","user_675aa5f7_162","user_67a30235_29","user_67e679ee_48","user_68c60e5d_189","user_698965a5_65","user_6dc959a7_148","user_6fc814f7_179","user_70fb4880_108","user_743bf19a_10","user_765a601b_141","user_7683ee52_137","user_76fda556_131","user_799d06c7_120","user_7a60e2b2_119","user_7b6f4034_121","user_7be9ba96_130","user_7c4c2597_24","user_8011c3dd_3","user_820130e7_173","user_82974c34_105","user_82fcf5e0_187","user_83029824_153","user_83f5bcf0_52","user_84ba801c_44","user_85e8e543_103","user_863d768f_142","user_8801804a_15","user_8a05a33b_159","user_8bca2d2d_63","user_8e3190be_175","user_8e8d73f4_46","user_8eb6c9f6_144","user_8fe3d788_89","user_90c8f079_37","user_950830e8_87","user_9600571a_193","user_963193e1_117","user_96cf2b03_136","user_97a25fad_59","user_983c4c6d_195","user_999c7504_26","user_9a559803_27","user_9af10d16_107","user_9dc9379d_51","user_a1df108c_151","user_a31002e0_22","user_a53baa6c_8","user_a581db79_39","user_a5cd9b0e_42","user_a6521bfe_123","user_a79f6107_166","user_a8331211_36","user_aa82f322_35","user_aa8aa143_69","user_aa8f8dcc_56","user_ad93177e_66","user_ae0736b0_60","user_aeeb1c6b_94","user_b0d004f8_199","user_b1406440_135","user_b1f2900b_185","user_b2f2603a_23","user_b32b9768_104","user_b35a7fe7_9","user_b38d191c_74","user_b41b09c5_161","user_b58a2c2d_194","user_b62ced4b_106","user_b67bd3c1_30","user_b6b1074c_4","user_b7b673b2_31","user_b80273a8_70","user_bd44a5df_167","user_beeff49a_146","user_c03b2f5f_172","user_c244c3d0_133","user_c24d40b8_84","user_c4c0e847_97","user_c82d5a93_20","user_c84b1e99_17","user_c8f7d225_180","user_cc0b66d0_28","user_cc395a2e_95","user_cd0461d0_155","user_cd411907_186","user_cf05b095_55","user_d0526957_72","user_d2f4183b_2","user_d424effc_54","user_d456c243_83","user_d477692c_58","user_d6d7b309_163","user_d7fad831_101","user_d8b908d3_85","user_da50a598_184","user_daaa2ca6_76","user_df464720_181","user_e17cecdb_129","user_e199d450_138","user_e2416269_19","user_e78b4580_145","user_e7977a3d_32","user_e7b63583_196","user_e8a7fecc_150","user_ec4987b5_0","user_f0a59b43_92","user_f4626453_7","user_f864b6eb_34","user_f93393d3_91","user_f98ae3c9_53","user_fada8efe_147","user_faedbd37_1","user_faf634f3_170","user_fbf3ac77_109","user_fd66a693_96","user_fd9d34cd_198","user_fed0f116_45"]}% 